# Setup

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json, glob, os, re
import numpy as np
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel, pearsonr
from scipy.io import loadmat
from IPython.display import display, Math, Latex

band_names = [r"$\delta$", r"$\theta$", r"$\alpha$", r"$\beta$", r"$\gamma$"]
sns.set_theme("paper", "ticks")

MAIN_DATA_FOLDER = (
    "/mnt/DATA/Giulio_NonLinearityResults/ReRun/"  # /path/to/some/folder"
)
RESULTS_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Results")
FIGURES_FOLDER = os.path.join(MAIN_DATA_FOLDER, "Figures")
if not os.path.isdir(FIGURES_FOLDER):
    os.mkdir(FIGURES_FOLDER)

In [ ]:
def coloured_box_kwargs(colour):
    return {
        "boxprops": {"color": colour},
        "widths":.6,
        "whiskerprops": {"color": colour},
        "flierprops": {"markeredgecolor": colour, "marker": "+"},
        "medianprops": {"lw": 1.6, "color": colour},
        "capprops": {"color": colour},
    }

In [ ]:
def boxplot(
    x, series, colours, percents, ax=None, ttest=False, variant="Holm-Bonferroni"
):
    if ax is not None:
        plt.sca(ax)
    STRETCH = len(series)
    OFFSET = -STRETCH / 2
    handles = []
    labels = []

    for data, colour in zip(series, colours):
        plt.boxplot(
            series[data],
            positions=np.arange(len(x)) * STRETCH + OFFSET,
            **coloured_box_kwargs(colour),
        )
        handles.append(Patch(facecolor="white", edgecolor=colour))
        labels.append(data)
        OFFSET += 1

    for percent, style in zip(percents, ["solid", "dashed", "dashdot", "dotted"]):
        plt.hlines(
            percent / 100,
            -STRETCH,
            (len(x) - 0.5) * STRETCH,
            "r",
            style,
            lw=1,
            zorder=1,
        )
        handles.append(Line2D([0], [0], lw=1, color="r", linestyle=style))
        labels.append(f"{percent}%")

    if ttest:
        if hasattr(ttest, "len") and len(ttest) == 2:
            first, second = ttest
        elif len(series) == 2:
            first, second = series.keys()
        significance = np.array(
            [
                ttest_rel(a, b, alternative="greater").pvalue
                for a, b in zip(series[first], series[second])
            ]
        )
        if variant == "Bonferroni":
            threshold = 0.05 / len(x)
        else:
            moving_threshold = 0.05 / (len(x) - np.arange(len(x)))
            significance_sorter = np.argsort(significance)
            above_threshold = significance[significance_sorter] > moving_threshold
            first_above = np.min(np.where(above_threshold))
            threshold = significance[significance_sorter[first_above]]

        markers = significance < threshold
        position = ((max(percents) / 100 if percents else 0) + plt.ylim()[1]) / 2
        ast = plt.scatter(
            np.arange(len(x))[markers] * STRETCH - OFFSET / 2,
            np.full(np.sum(markers), position, dtype=float),
            marker="*",
            color="k",
        )
        handles.append(ast)
        labels.append("Significative\ndifference")

    tickPos = np.arange(len(x)) * STRETCH
    plt.xticks(tickPos, x)
    plt.xlim((-STRETCH, (len(x) - 0.5) * STRETCH))
    plt.legend(handles, labels)
    return plt.gca()

# Amount of non-linearity
## fMRI

In [ ]:
regions = [
    10,
    30,
    50,
    70,
    100,
    150,
    200,
    230,
    270,
    300,
    350,
    400,
    450,
    500,
    550,
    600,
    650,
    700,
    750,
    800,
    850,
    900,
    950,
]
fMRI_results = {"True": [], "Shadow": []}
for region in regions:
    out_name = f"clean_cra_strin_{region}_bin7"
    with open(
        os.path.join(RESULTS_FOLDER, out_name, out_name + "_globalStats.json")
    ) as fp:
        tmp = {k: np.array(v) for k, v in json.load(fp).items()}
    fMRI_results["True"].append(1 - tmp["gaussMI"] / tmp["totalMI"])
    fMRI_results["Shadow"].append(1 - tmp["gaussMIshadow"] / tmp["totalMIshadow"])

In [ ]:
fig = plt.figure(figsize=(12,6))
boxplot(regions, fMRI_results, ["darkorange", "blue"], [0, 5], plt.gca(), True)
plt.xlabel("# regions")
plt.ylabel("RNL")
plt.show()

## fMRI test-retest

## EEG

## iEEG

## Single unit spikes

# Localisation of non-linearities
## fMRI

## EEG

# Sources of non-linearity
## fMRI

## EEG

## iEEG

## Single unit spikes

# Reliability of non-linearity
## fMRI test-retest

## EEG

## iEEG